IMPORTING DATA

In [1]:
import pandas as pd
import numpy as np

df1=pd.read_csv("economic_full_1.csv")
df2 = pd.read_csv("weather_data.csv", encoding='latin-1', skiprows=3, header=0)
df3=pd.read_csv("PGCB_date_power_demand.csv")

In [2]:
df3

,datetime,generation_mw,demand_mw,load_shedding,gas,liquid_fuel,coal,hydro,solar,wind,india_bheramara_hvdc,india_tripura,india_adani,nepal,remarks
0,2015-04-19 22:00:00,6323.0,6323,0,0,0,0,0,NaN,NaN,0,0,NaN,NaN,NaN
1,2015-04-19 21:00:00,6667.0,6667,0,0,0,0,0,NaN,NaN,0,0,NaN,NaN,NaN
2,2015-04-19 19:00:00,6897.0,6897,0,4415,1836,161,41,NaN,NaN,444,0,NaN,NaN,NaN
3,2015-04-19 18:30:00,6933.0,6933,0,4423,1862,159,45,NaN,NaN,444,0,NaN,NaN,Evening_Peak
4,2015-04-19 18:00:00,6874.0,6874,0,4319,1892,155,65,NaN,NaN,443,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92645,2025-06-17 04:00:00,12698.0,12698,0,6030,834,3610,172,0.0,42.0,924,38,1010.0,38.0,NaN
92646,2025-06-17 03:00:00,13280.0,13280,0,6347,999,3698,172,0.0,40.0,924,38,1024.0,38.0,NaN
92647,2025-06-17 02:00:00,13719.0,13719,0,6340,1196,3881,172,0.0,27.0,924,36,1105.0,38.0,NaN
92648,2025-06-17 01:00:00,14093.0,14115,21,6323,1350,4019,172,0.0,28.0,924,38,1201.0,38.0,NaN


In [3]:
print(df3.isnull().sum())

datetime                    0
generation_mw               0
demand_mw                   0
load_shedding               0
gas                         0
liquid_fuel                 0
coal                        0
hydro                       0
solar                   22133
wind                    73974
india_bheramara_hvdc        0
india_tripura               0
india_adani             85312
nepal                   87299
remarks                 86257
dtype: int64


In [4]:
df3 = df3.drop(columns=['remarks', 'nepal', 'india_adani', 'wind'])

In [5]:
df3['datetime'] = pd.to_datetime(df3['datetime'])
df3 = df3.sort_values('datetime').reset_index(drop=True)

In [6]:

print(f"Duplicate timestamps: {df3.duplicated(subset='datetime').sum()}")
print(df3[df3.duplicated(subset='datetime', keep=False)].head(5))

Duplicate timestamps: 432
               datetime  generation_mw  demand_mw  load_shedding   gas  \
76  2015-04-22 21:00:00         5811.0       5811              0  4158   
77  2015-04-22 21:00:00         5368.0       5368              0  4184   
79  2015-04-23 00:00:00         5726.0       5726              0  4199   
80  2015-04-23 00:00:00         3976.0       3976              0  3116   
158 2015-04-27 00:00:00         5236.0       5236              0  4417   

     liquid_fuel  coal  hydro  solar  india_bheramara_hvdc  india_tripura  
76          1021   162     30    NaN                   440              0  
77           552   160     30    NaN                   442              0  
79           790   158     46    NaN                   433              0  
80           230   158     30    NaN                   442              0  
158          159   162     46    NaN                   452              0  


In [7]:

df3 = df3.groupby('datetime').mean().reset_index()


In [8]:

df3['minute'] = df3['datetime'].dt.minute
print(df3['minute'].value_counts())
print(df3[df3['minute'] != 0][['datetime', 'demand_mw']].head(10))

minute
0     88046
30     4172
Name: count, dtype: int64
               datetime  demand_mw
18  2015-04-19 17:30:00     6299.0
20  2015-04-19 18:30:00     6933.0
46  2015-04-21 17:30:00     6698.0
48  2015-04-21 18:30:00     7409.0
71  2015-04-22 17:30:00     6195.0
73  2015-04-22 18:30:00     6689.0
96  2015-04-23 17:30:00     6141.0
98  2015-04-23 18:30:00     6710.0
122 2015-04-24 17:30:00     5518.0
124 2015-04-24 18:30:00     6678.0


In [9]:

df3 = df3.drop(columns=['minute'])
df3['datetime'] = df3['datetime'].dt.floor('h')
ncols = df3.select_dtypes(include='number').columns.tolist()
df3 = df3.groupby('datetime')[ncols].mean().reset_index()


In [10]:

full_range = pd.date_range(start=df3['datetime'].min(), 
                           end=df3['datetime'].max(), 
                           freq='h')

print(f"Total expected hourly timestamps: {len(full_range)}")
print(f"Current timestamps: {len(df3)}")
print(f"Missing hours: {len(full_range) - len(df3)}")


df3 = df3.set_index('datetime').reindex(full_range).reset_index()
df3 = df3.rename(columns={'index': 'datetime'})

print(df3.isnull().sum())

Total expected hourly timestamps: 89101
Current timestamps: 88050
Missing hours: 1051
datetime                    0
generation_mw            1051
demand_mw                1051
load_shedding            1051
gas                      1051
liquid_fuel              1051
coal                     1051
hydro                    1051
solar                   21571
india_bheramara_hvdc     1051
india_tripura            1051
dtype: int64


In [11]:

df3 = df3.set_index('datetime')

cols_to_interpolate = ['generation_mw', 'demand_mw', 'load_shedding', 
                       'gas', 'liquid_fuel', 'coal', 'hydro', 
                       'india_bheramara_hvdc', 'india_tripura', 'solar']

for col in cols_to_interpolate:
    df3[col] = df3[col].interpolate(method='time')

df3 = df3.reset_index()

print(df3.isnull().sum())
print(df3.shape)

datetime                    0
generation_mw               0
demand_mw                   0
load_shedding               0
gas                         0
liquid_fuel                 0
coal                        0
hydro                       0
solar                   21246
india_bheramara_hvdc        0
india_tripura               0
dtype: int64
(89101, 11)


In [12]:

df3['solar'] = df3['solar'].fillna(0)
print(df3.isnull().sum())

datetime                0
generation_mw           0
demand_mw               0
load_shedding           0
gas                     0
liquid_fuel             0
coal                    0
hydro                   0
solar                   0
india_bheramara_hvdc    0
india_tripura           0
dtype: int64


In [13]:
df3

,datetime,generation_mw,demand_mw,load_shedding,gas,liquid_fuel,coal,hydro,solar,india_bheramara_hvdc,india_tripura
0,2015-04-19 00:00:00,4821.0,4821.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2015-04-19 01:00:00,3612.0,3612.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2015-04-19 02:00:00,3727.0,3727.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2015-04-19 03:00:00,3632.0,3632.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2015-04-19 04:00:00,3641.0,3641.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
89096,2025-06-17 08:00:00,11896.0,11896.0,0.0,5678.0,490.0,3470.0,132.0,219.0,924.0,24.0
89097,2025-06-17 09:00:00,12290.0,12290.0,0.0,6065.0,531.0,3491.0,132.0,203.0,924.0,24.0
89098,2025-06-17 10:00:00,12443.0,12443.0,0.0,5900.0,552.0,3601.0,172.0,215.0,923.0,26.0
89099,2025-06-17 11:00:00,12826.0,12826.0,0.0,5896.0,595.0,3591.0,172.0,324.0,924.0,26.0


FINDING OUTLIERS

In [14]:
from scipy import stats

ncols = ['generation_mw', 'demand_mw', 'load_shedding', 
                'gas', 'liquid_fuel', 'coal', 'hydro', 
                'solar', 'india_bheramara_hvdc', 'india_tripura']

for col in ncols:
    z_scores = np.abs(stats.zscore(df3[col]))
    outliers = (z_scores > 3).sum()
    print(f"{col:25} → outliers: {outliers:5}  max: {df3[col].max():.1f}  mean: {df3[col].mean():.1f}  std: {df3[col].std():.1f}")

generation_mw             → outliers:     1  max: 64526500.0  mean: 9395.2  std: 216156.0
demand_mw                 → outliers:    23  max: 121000.0  mean: 8756.7  std: 2731.3
load_shedding             → outliers:  1121  max: 65359.0  mean: 81.6  std: 448.0
gas                       → outliers:   117  max: 74818.0  mean: 5114.3  std: 1137.0
liquid_fuel               → outliers:     3  max: 29222897.0  mean: 2001.8  std: 97975.5
coal                      → outliers:   747  max: 31687.0  mean: 970.6  std: 1222.1
hydro                     → outliers:    91  max: 5623.0  mean: 96.4  std: 71.5
solar                     → outliers:  3063  max: 2998.0  mean: 36.6  std: 96.5
india_bheramara_hvdc      → outliers:    19  max: 76292.0  mean: 661.0  std: 353.7
india_tripura             → outliers:    49  max: 1234.0  mean: 93.3  std: 46.5


CLIPPING

In [15]:

cols_to_clip = ['generation_mw', 'liquid_fuel', 'india_bheramara_hvdc', 
                'demand_mw', 'coal', 'gas', 
                'hydro', 'india_tripura']

for col in cols_to_clip:
    Q1 = df3[col].quantile(0.25)
    Q3 = df3[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = max(0, Q1 - 1.5 * IQR)
    upper = Q3 + 1.5 * IQR
    df3[col] = df3[col].clip(lower=lower, upper=upper)


In [16]:
df2

,time,temperature_2m (°C),relative_humidity_2m (%),apparent_temperature (°C),precipitation (mm),dew_point_2m (°C),soil_temperature_0_to_7cm (°C),wind_direction_10m (°),cloud_cover (%),sunshine_duration (s)
0,2014-01-01T00:00,13.9,89,13.3,0.0,12.1,16.4,313,0,0.0
1,2014-01-01T01:00,13.6,91,13.2,0.0,12.1,16.0,317,0,0.0
2,2014-01-01T02:00,13.3,91,12.8,0.0,11.9,15.7,317,0,0.0
3,2014-01-01T03:00,13.0,92,12.5,0.0,11.8,15.4,319,0,0.0
4,2014-01-01T04:00,12.7,93,12.2,0.0,11.6,15.2,322,0,0.0
...,...,...,...,...,...,...,...,...,...,...
107299,2026-03-29T19:00,25.5,81,29.0,0.0,22.0,30.1,87,10,0.0
107300,2026-03-29T20:00,25.5,82,29.4,0.0,22.1,28.9,163,25,0.0
107301,2026-03-29T21:00,25.0,83,29.0,0.0,22.0,27.9,203,16,0.0
107302,2026-03-29T22:00,24.5,87,28.8,0.0,22.0,27.1,153,5,0.0


In [17]:
print(df2.shape)
print(df2.dtypes)
print(df2.isnull().sum())
print(df2.describe())

(107304, 10)
time                               object
temperature_2m (°C)               float64
relative_humidity_2m (%)            int64
apparent_temperature (°C)         float64
precipitation (mm)                float64
dew_point_2m (°C)                 float64
soil_temperature_0_to_7cm (°C)    float64
wind_direction_10m (°)              int64
cloud_cover (%)                     int64
sunshine_duration (s)             float64
dtype: object
time                              0
temperature_2m (°C)               0
relative_humidity_2m (%)          0
apparent_temperature (°C)         0
precipitation (mm)                0
dew_point_2m (°C)                 0
soil_temperature_0_to_7cm (°C)    0
wind_direction_10m (°)            0
cloud_cover (%)                   0
sunshine_duration (s)             0
dtype: int64
       temperature_2m (°C)  relative_humidity_2m (%)  \
count        107304.000000             107304.000000   
mean             25.457445                 76.806233   
std         

In [18]:
df2['time'] = pd.to_datetime(df2['time'])
df2 = df2.rename(columns={'time': 'datetime'})

print(f"Duplicate timestamps: {df2.duplicated(subset='datetime').sum()}")

Duplicate timestamps: 0


In [19]:
df1.head()

,Country Name,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,1966,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,X,"Intentional homicides, male (per 100,000 male)",VC.IHR.PSRC.MA.P5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,X,Battle-related deaths (number of people),VC.BTL.DETH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,41.000000,47.000000,NaN,NaN,2.000000,NaN,2.000000,NaN,2.000000,NaN
2,X,Voice and Accountability: Percentile Rank,VA.PER.RNK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,30.541872,30.049261,27.184465,26.570047,26.570047,28.019323,28.019323,27.450981,NaN,NaN
3,X,Transport services (% of commercial service ex...,TX.VAL.TRAN.ZS.WT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,20.820301,21.484188,22.048898,19.559759,22.257010,27.162699,25.520411,17.176637,23.495141,NaN
4,X,"Computer, communications and other services (%...",TX.VAL.OTHR.ZS.WT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,64.068305,57.202109,62.313903,63.578283,67.395621,64.565856,63.537174,67.817057,65.566600,NaN


In [20]:
np.array(df1[df1['Indicator Name'].str.contains('GDP|Industry|Electricity', case=False)]["Indicator Name"])

array(['Government expenditure per student, tertiary (% of GDP per capita)',
       'Government expenditure per student, primary (% of GDP per capita)',
       'Net lending (+) / net borrowing (-) (% of GDP)',
       'Net incurrence of liabilities, total (% of GDP)',
       'Net acquisition of financial assets (% of GDP)',
       'Domestic credit to private sector (% of GDP)',
       'Nitrous oxide (N2O) emissions from Power Industry (Energy) (Mt CO2e)',
       'Carbon intensity of GDP (kg CO2e per constant 2015 US$ of GDP)',
       'Industry (including construction), value added (annual % growth)',
       'Manufacturing, value added (% of GDP)', 'Trade (% of GDP)',
       'Gross fixed capital formation (% of GDP)',
       'Exports of goods and services (% of GDP)',
       'Final consumption expenditure (% of GDP)',
       'Services, value added (% of GDP)',
       'Industry (including construction), value added (constant 2015 US$)',
       'Agriculture, forestry, and fishing, value ad

In [21]:
import pandas as pd

indicators = [
    'GDP (constant 2015 US$)',
    'Industry (including construction), value added (constant 2015 US$)',
    'Manufacturing, value added (% of GDP)',
    'Services, value added (% of GDP)',
    'Energy intensity level of primary energy (MJ/$2021 PPP GDP)',
    'Access to electricity (% of population)',
    'Access to electricity, rural (% of rural population)'
]

df1= df1[df1['Indicator Name'].isin(indicators)].copy()

df= df1.drop(columns=['Country Name', 'Indicator Code'])

# wide format
dfm= df.melt(
    id_vars=['Indicator Name'], 
    var_name='Year', 
    value_name='Value'
)

#YEAR SHOULD BE AN INTEGER
dfm['Year'] = pd.to_numeric(dfm['Year'], errors='coerce')
dfm= dfm.dropna(subset=['Year'])
dfm['Year'] = dfm['Year'].astype(int)

#COVERTING YEARS INTO COLOUMNS
df1= dfm.pivot(
    index='Year', 
    columns='Indicator Name', 
    values='Value'
).reset_index()
#REMOVIE INDICATOR NAME COLUMN
df1.columns.name = None


print(df1.shape)
display(df1.tail(10))

(66, 8)


,Year,Access to electricity (% of population),"Access to electricity, rural (% of rural population)",Energy intensity level of primary energy (MJ/$2021 PPP GDP),GDP (constant 2015 US$),"Industry (including construction), value added (constant 2015 US$)","Manufacturing, value added (% of GDP)","Services, value added (% of GDP)"
56,2016,75.9,66.1,2.20,2.090280e+11,5.816930e+10,20.347948,51.207804
57,2017,88.0,81.6,2.12,2.228040e+11,6.298116e+10,20.075062,51.383901
58,2018,86.9,81.3,1.96,2.391120e+11,6.940260e+10,20.802296,50.890161
59,2019,92.2,88.9,1.95,2.579580e+11,7.747643e+10,21.207944,50.849289
60,2020,96.2,95.2,1.90,2.668530e+11,8.027416e+10,20.598332,51.511309
61,2021,99.0,98.5,1.93,2.853690e+11,8.853206e+10,21.235673,51.301507
62,2022,99.4,99.2,NaN,3.056300e+11,9.725748e+10,21.764819,51.038083
63,2023,99.5,99.6,NaN,3.232800e+11,1.053950e+11,22.342019,51.111913
64,2024,NaN,NaN,NaN,3.369330e+11,1.090910e+11,21.892960,51.416655
65,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:

pd.options.display.float_format = '{:,.2f}'.format

df_temp = pd.DataFrame({
    'NaN_Count': df1.isna().sum(),
    'Min': df1.min(numeric_only=True),
    'Max': df1.max(numeric_only=True),
    'Mean': df1.mean(numeric_only=True)
})

display(df_temp)

,NaN_Count,Min,Max,Mean
Year,0,"1,960.00","2,025.00","1,992.50"
Access to electricity (% of population),33,9.91,99.50,52.33
"Access to electricity, rural (% of rural population)",35,2.24,99.60,44.11
Energy intensity level of primary energy (MJ/$2021 PPP GDP),44,1.90,2.58,2.26
GDP (constant 2015 US$),1,"22,255,159,274.00","336,933,000,000.00","95,160,572,754.91"
"Industry (including construction), value added (constant 2015 US$)",1,"1,896,402,129.00","109,091,000,000.00","22,816,833,162.54"
"Manufacturing, value added (% of GDP)",1,3.98,22.34,13.26
"Services, value added (% of GDP)",1,26.43,53.71,44.90


In [23]:

df1= df1.interpolate(method='linear', limit_direction='both')

df1= df1.ffill().bfill()

print("--- Remaining NaN Count ---")
print(df1.isna().sum())

display(df1.tail(10))

--- Remaining NaN Count ---
Year                                                                  0
Access to electricity (% of population)                               0
Access to electricity, rural (% of rural population)                  0
Energy intensity level of primary energy (MJ/$2021 PPP GDP)           0
GDP (constant 2015 US$)                                               0
Industry (including construction), value added (constant 2015 US$)    0
Manufacturing, value added (% of GDP)                                 0
Services, value added (% of GDP)                                      0
dtype: int64


,Year,Access to electricity (% of population),"Access to electricity, rural (% of rural population)",Energy intensity level of primary energy (MJ/$2021 PPP GDP),GDP (constant 2015 US$),"Industry (including construction), value added (constant 2015 US$)","Manufacturing, value added (% of GDP)","Services, value added (% of GDP)"
56,2016,75.90,66.10,2.20,"209,028,000,000.00","58,169,298,008.00",20.35,51.21
57,2017,88.00,81.60,2.12,"222,804,000,000.00","62,981,158,233.00",20.08,51.38
58,2018,86.90,81.30,1.96,"239,112,000,000.00","69,402,599,931.00",20.80,50.89
59,2019,92.20,88.90,1.95,"257,958,000,000.00","77,476,428,865.00",21.21,50.85
60,2020,96.20,95.20,1.90,"266,853,000,000.00","80,274,163,392.00",20.60,51.51
61,2021,99.00,98.50,1.93,"285,369,000,000.00","88,532,058,911.00",21.24,51.30
62,2022,99.40,99.20,1.93,"305,630,000,000.00","97,257,480,670.00",21.76,51.04
63,2023,99.50,99.60,1.93,"323,280,000,000.00","105,395,000,000.00",22.34,51.11
64,2024,99.50,99.60,1.93,"336,933,000,000.00","109,091,000,000.00",21.89,51.42
65,2025,99.50,99.60,1.93,"336,933,000,000.00","109,091,000,000.00",21.89,51.42


MERGING THE THREE
 DATASETS

In [24]:
import pandas as pd

df2['datetime'] = pd.to_datetime(df2['datetime'])
df3['datetime'] = pd.to_datetime(df3['datetime'])

# how=inner makes SURE THAT ONLY THE EXACTLY OVERLAPPING DATE TIME ARE TAKEN
mdf = pd.merge(df3, df2, on='datetime', how='inner')

mdf['Year'] = mdf['datetime'].dt.year

# Using how='left' ensures that the df1 that is the left dataframe gets priority while merging
fdf = pd.merge(mdf, df1, on='Year', how='left')

# 5. Verify the final structure
print(fdf.shape)
display(fdf.head())


(89101, 28)


,datetime,generation_mw,demand_mw,load_shedding,gas,liquid_fuel,coal,hydro,solar,india_bheramara_hvdc,...,cloud_cover (%),sunshine_duration (s),Year,Access to electricity (% of population),"Access to electricity, rural (% of rural population)",Energy intensity level of primary energy (MJ/$2021 PPP GDP),GDP (constant 2015 US$),"Industry (including construction), value added (constant 2015 US$)","Manufacturing, value added (% of GDP)","Services, value added (% of GDP)"
0,2015-04-19 00:00:00,"4,821.00","4,821.00",0.00,"2,311.00",0.00,0.00,0.00,0.00,0.00,...,9,0.00,2015,74.00,63.80,2.17,"195,147,000,000.00","52,360,557,158.00",16.79,53.71
1,2015-04-19 01:00:00,"3,612.00","3,612.00",0.00,"2,311.00",0.00,0.00,0.00,0.00,0.00,...,26,0.00,2015,74.00,63.80,2.17,"195,147,000,000.00","52,360,557,158.00",16.79,53.71
2,2015-04-19 02:00:00,"3,727.00","3,727.00",0.00,"2,311.00",0.00,0.00,0.00,0.00,0.00,...,32,0.00,2015,74.00,63.80,2.17,"195,147,000,000.00","52,360,557,158.00",16.79,53.71
3,2015-04-19 03:00:00,"3,632.00","3,632.00",0.00,"2,311.00",0.00,0.00,0.00,0.00,0.00,...,30,0.00,2015,74.00,63.80,2.17,"195,147,000,000.00","52,360,557,158.00",16.79,53.71
4,2015-04-19 04:00:00,"3,641.00","3,641.00",0.00,"2,311.00",0.00,0.00,0.00,0.00,0.00,...,67,0.00,2015,74.00,63.80,2.17,"195,147,000,000.00","52,360,557,158.00",16.79,53.71


ADDING LAG FEATURES

In [25]:

fdf = fdf.sort_values(by='datetime')

#LAST HOUR DEMAND COLUMN
fdf['demand_prev_hour'] = fdf['demand_mw'].shift(1)
fdf = fdf.dropna(subset=['demand_prev_hour'])


# DEMAND 24 HOURS AGO, THAT IS YESTERDAY AT THIS TIME, WHAT WAS THE DEMAND
fdf['demand_prev_day_same_hour'] = fdf['demand_mw'].shift(24)
fdf = fdf.sort_values(by='datetime')
fdf['demand_prev_day_same_hour'] = fdf['demand_mw'].shift(24)
fdf = fdf.dropna(subset=['demand_prev_day_same_hour'])


display(fdf[['datetime', 'demand_mw', 'demand_prev_hour', 'demand_prev_day_same_hour']].head(5))

,datetime,demand_mw,demand_prev_hour,demand_prev_day_same_hour
25,2015-04-20 01:00:00,"5,607.29","5,678.00","3,612.00"
26,2015-04-20 02:00:00,"5,536.57","5,607.29","3,727.00"
27,2015-04-20 03:00:00,"5,465.86","5,536.57","3,632.00"
28,2015-04-20 04:00:00,"5,395.14","5,465.86","3,641.00"
29,2015-04-20 05:00:00,"5,324.43","5,395.14","3,283.00"


In [26]:
fdf.columns.size

30

SPLITTING DATA FOR TRAINING AND TESTING

In [27]:

train_df = fdf[fdf['Year'] <= 2023].copy()
test_df = fdf[fdf['Year'] == 2024].copy()


MODEL

In [28]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_percentage_error
import matplotlib.pyplot as plt

target = 'demand_mw'

#separate the features to be inputed to the model in a list
features = [col for col in train_df.columns if col not in [target, 'datetime']]

# 3. Create X(features) and y(demand) for both training and testing
X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

print(len(features))

28


In [29]:
xgb_model = XGBRegressor(
    n_estimators=500,        # Number of trees
    learning_rate=0.05,      # Speed of learning
    max_depth=6,             # Maximum depth of each tree
    subsample=0.8,           # Use 80% of data per tree, low overfitting
    colsample_bytree=0.8,    # Use 80% of features per tree
    random_state=42,         # For reproducibility
    n_jobs=-1                # Use all CPU cores for faster training
)


xgb_model.fit(X_train, y_train)
print("Training Complete!")

Training Complete!


In [30]:

predictions = xgb_model.predict(X_test)

# Calculate MAPE
mape_score = mean_absolute_percentage_error(y_test, predictions) * 100


print(f"XGBoost MAPE: {mape_score:.2f}%")

XGBoost MAPE: 2.85%


SAMPLE TESTS ON 10 RANDOM SAMPLES FROM TEST DATA

In [31]:
sample_indices = X_test.sample(10).index
X_sample = X_test.loc[sample_indices]
y_true = y_test.loc[sample_indices]
dates = test_df.loc[sample_indices, 'datetime']
y_pred = xgb_model.predict(X_sample)
comparison_df = pd.DataFrame({
    'Datetime': dates,
    'Actual Demand (MW)': y_true,
    'Predicted Demand (MW)': y_pred
})
comparison_df['Error (MW)'] = comparison_df['Predicted Demand (MW)'] - comparison_df['Actual Demand (MW)']
comparison_df['% Error'] = (abs(comparison_df['Error (MW)']) / comparison_df['Actual Demand (MW)']) * 100
pd.options.display.float_format = '{:,.2f}'.format
display(comparison_df.sort_values(by='Datetime'))

,Datetime,Actual Demand (MW),Predicted Demand (MW),Error (MW),% Error
78492,2024-04-01 12:00:00,"13,050.00","12,429.25",-620.75,4.76
79197,2024-04-30 21:00:00,"16,321.50","15,373.52",-947.98,5.81
80037,2024-06-04 21:00:00,"14,900.00","14,055.71",-844.29,5.67
81007,2024-07-15 07:00:00,"13,300.00","13,303.98",3.98,0.03
81449,2024-08-02 17:00:00,"9,462.00","9,378.59",-83.41,0.88
81821,2024-08-18 05:00:00,"11,270.00","11,025.52",-244.48,2.17
82316,2024-09-07 20:00:00,"15,600.00","15,271.80",-328.20,2.10
83580,2024-10-30 12:00:00,"12,600.00","12,511.04",-88.96,0.71
83737,2024-11-06 01:00:00,"9,397.00","9,374.98",-22.02,0.23
84989,2024-12-28 05:00:00,"6,884.00","6,838.49",-45.51,0.66
